# AdrenalMNIST3D — mesh generation

Downloads AdrenalMNIST3D from [MedMNIST](https://medmnist.com/), runs marching cubes on each
segmentation mask, aligns the mesh center-of-mass to the origin, and saves `.off` files.

**Dataset:** 1,584 left/right adrenal glands (792 patients).  
Each 3D shape is a binary mask resized from a 64 mm × 64 mm × 64 mm crop to 28 × 28 × 28 voxels.  
Labels: `0` = normal, `1` = hyperplasia.  
Split: train 1,188 / val 98 / test 298.

Output is written to `dataset/medmnist/adrenal3d/` under the project root.

In [1]:
from medmnist import AdrenalMNIST3D
import numpy as np
from skimage.measure import marching_cubes
import os
import shutil


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/yk/anaconda3/envs/pykarambola_helper/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/yk/anaconda3/envs/pykarambola_helper/lib/python3.11/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/Users/yk/anaconda3/envs/pykarambola_helper/lib/python3.11/site-packages/ipykernel/kernelapp.py", 

## Load datasets

In [ ]:
train_dataset = AdrenalMNIST3D(split="train", download=True)
val_dataset   = AdrenalMNIST3D(split="val",   download=True)
test_dataset  = AdrenalMNIST3D(split="test",  download=True)

train_dataset_64 = AdrenalMNIST3D(split="train", download=True, size=64)
val_dataset_64   = AdrenalMNIST3D(split="val",   download=True, size=64)
test_dataset_64  = AdrenalMNIST3D(split="test",  download=True, size=64)

Using downloaded and verified file: /Users/yk/.medmnist/adrenalmnist3d.npz
Using downloaded and verified file: /Users/yk/.medmnist/adrenalmnist3d.npz
Using downloaded and verified file: /Users/yk/.medmnist/adrenalmnist3d.npz
Using downloaded and verified file: /Users/yk/.medmnist/adrenalmnist3d_64.npz
Using downloaded and verified file: /Users/yk/.medmnist/adrenalmnist3d_64.npz
Using downloaded and verified file: /Users/yk/.medmnist/adrenalmnist3d_64.npz


In [3]:
print(train_dataset)

Dataset AdrenalMNIST3D of size 28 (adrenalmnist3d)
    Number of datapoints: 1188
    Root location: /Users/yk/.medmnist
    Split: train
    Task: binary-class
    Number of channels: 1
    Meaning of labels: {'0': 'normal', '1': 'hyperplasia'}
    Number of samples: {'train': 1188, 'val': 98, 'test': 298}
    Description: The AdrenalMNIST3D is a new 3D shape classification dataset, consisting of shape masks from 1,584 left and right adrenal glands (i.e., 792 patients). Collected from Zhongshan Hospital Affiliated to Fudan University, each 3D shape of adrenal gland is annotated by an expert endocrinologist using abdominal computed tomography (CT), together with a binary classification label of normal adrenal gland or adrenal mass. Considering patient privacy, we do not provide the source CT scans, but the real 3D shapes of adrenal glands and their classification labels. We calculate the center of adrenal and resize the center-cropped 64mm×64mm×64mm volume into 28×28×28. The dataset is

## Helper functions

In [7]:
def mesh_center_of_mass(verts, faces):
    """Volume-weighted centroid of a closed triangle mesh via the divergence theorem.

    Equivalent to trimesh.center_mass. Each triangle is treated as the base of
    a tetrahedron meeting at the origin; the centroid is the volume-weighted
    mean of tetrahedron centroids (v0+v1+v2)/4.
    """
    v0 = verts[faces[:, 0]]
    v1 = verts[faces[:, 1]]
    v2 = verts[faces[:, 2]]
    d = np.einsum('ij,ij->i', v0, np.cross(v1 - v0, v2 - v0))
    d_sum = d.sum()
    if abs(d_sum) < 1e-12:
        return np.zeros(3)
    return np.einsum('i,ij->j', d, v0 + v1 + v2) / (4.0 * d_sum)


def save_off(path, verts, faces):
    """Write a triangle mesh to an OFF file without external dependencies."""
    with open(path, 'w') as f:
        f.write("OFF\n")
        f.write(f"{len(verts)} {len(faces)} 0\n")
        for v in verts:
            f.write(f"{v[0]:.6f} {v[1]:.6f} {v[2]:.6f}\n")
        for face in faces:
            f.write(f"3 {face[0]} {face[1]} {face[2]}\n")


def process_and_save(dataset, output_folder, split_name,
                     padding=((1, 1), (1, 1), (1, 1))):
    """Run marching cubes on each sample, align CoM to origin, save as OFF.

    Parameters
    ----------
    dataset       : AdrenalMNIST3D split
    output_folder : root output directory
    split_name    : 'train', 'validation', or 'test'
    padding       : zero-padding applied before marching cubes to close open surfaces
    """
    out_dir = os.path.join(output_folder, split_name)
    os.makedirs(out_dir, exist_ok=True)

    for idx in range(len(dataset)):
        seg = dataset.imgs[idx]
        label = dataset.labels[idx][0]

        padded = np.pad(seg, padding, mode='constant', constant_values=0)

        verts, faces, _, _ = marching_cubes(padded, spacing=(1, 1, 1),
                                            gradient_direction='ascent')

        # Align mesh center-of-mass to origin via the divergence theorem
        com = mesh_center_of_mass(verts, faces)
        verts = verts - com

        out_path = os.path.join(out_dir, f"{split_name}_image_{idx}_label_{label}.off")
        save_off(out_path, verts, faces)

## Generate meshes — 28 × 28 × 28

In [8]:
output_folder_28 = "../../dataset/medmnist_benchmarking/adrenal3d/adrenal3d_28"

if os.path.exists(output_folder_28):
    shutil.rmtree(output_folder_28)
os.makedirs(output_folder_28)

for dataset, split in [(train_dataset, 'train'),
                        (val_dataset,   'validation'),
                        (test_dataset,  'test')]:
    process_and_save(dataset, output_folder_28, split)
    print(f"  {split}: {len(dataset)} meshes saved")

print("Done — adrenal3d_28")

  train: 1188 meshes saved
  validation: 98 meshes saved
  test: 298 meshes saved
Done — adrenal3d_28


## Generate meshes — 64 × 64 × 64

In [9]:
output_folder_64 = "../../dataset/medmnist_benchmarking/adrenal3d/adrenal3d_64"

if os.path.exists(output_folder_64):
    shutil.rmtree(output_folder_64)
os.makedirs(output_folder_64)

for dataset, split in [(train_dataset_64, 'train'),
                        (val_dataset_64,   'validation'),
                        (test_dataset_64,  'test')]:
    process_and_save(dataset, output_folder_64, split)
    print(f"  {split}: {len(dataset)} meshes saved")

print("Done — adrenal3d_64")

  train: 1188 meshes saved
  validation: 98 meshes saved
  test: 298 meshes saved
Done — adrenal3d_64
